# Azure VM Costs

This notebook fetches Azure VM pricing using the **public Retail Prices API** (no credentials required).

**Data Source:** Azure Retail Prices API  
**Output:** `vm_costs` table with cloud = 'AZURE' (on_demand, spot, reserved_1y, reserved_3y)


In [ ]:
# Configuration
CATALOG = "lakemeter_catalog"
SCHEMA = "lakemeter"
TABLE_INSTANCE_RATES = f"{CATALOG}.{SCHEMA}.instance_rates"
TABLE_VM_COSTS = f"{CATALOG}.{SCHEMA}.vm_costs"
TABLE_SKU_REGION_MAPPING = f"{CATALOG}.{SCHEMA}.sku_region_mapping"

# Load Azure regions from sku_region_mapping (SOURCE OF TRUTH from dbu_prices)
# This ensures we fetch VM costs for ALL regions that have DBU pricing
regions_df = spark.sql(f"""
    SELECT DISTINCT region_code 
    FROM {TABLE_SKU_REGION_MAPPING} 
    WHERE UPPER(cloud) = 'AZURE'
""").toPandas()

AZURE_REGIONS = regions_df['region_code'].tolist()

print(f"✅ Config: {CATALOG}.{SCHEMA}")
print(f"📍 Loaded {len(AZURE_REGIONS)} Azure regions from sku_region_mapping (dbu_prices source)")
print(f"   Regions: {AZURE_REGIONS}")


In [ ]:
# Get Azure instance types from instance_rates table
instance_types_df = spark.sql(f"""
    SELECT DISTINCT instance_type 
    FROM {TABLE_INSTANCE_RATES} 
    WHERE cloud = 'AZURE'
""").toPandas()

INSTANCE_TYPES_RAW = instance_types_df['instance_type'].tolist()

# Normalize instance types - create a lookup set with multiple formats
# Azure has many naming variations:
#   - DS1_v2 vs D1s_v2 (premium storage notation)
#   - F4s vs F4_s (with/without underscore)
#   - Standard_ prefix present or absent

INSTANCE_TYPES_LOOKUP = set()
INSTANCE_TO_CANONICAL = {}  # Map API name back to our canonical name

def add_variations(canonical: str, variant: str):
    """Add a lookup variation and map it back to canonical name."""
    variant_lower = variant.lower()
    INSTANCE_TYPES_LOOKUP.add(variant_lower)
    INSTANCE_TO_CANONICAL[variant_lower] = canonical

for inst in INSTANCE_TYPES_RAW:
    inst_clean = inst.strip()
    
    # Base form (lowercase)
    add_variations(inst_clean, inst_clean)
    
    # Without Standard_ prefix
    no_prefix = inst_clean.replace("Standard_", "").replace("standard_", "")
    add_variations(inst_clean, no_prefix)
    add_variations(inst_clean, f"standard_{no_prefix.lower()}")
    
    # Handle DS_v2 <-> D_s_v2 naming variations
    # Azure API sometimes uses "D1s_v2" instead of "DS1_v2"
    import re
    
    # Pattern: DS1_v2 -> D1s_v2, DS12_v2 -> D12s_v2
    ds_match = re.match(r'^(Standard_)?DS(\d+)(_v\d+)?$', inst_clean, re.IGNORECASE)
    if ds_match:
        prefix = ds_match.group(1) or ''
        num = ds_match.group(2)
        ver = ds_match.group(3) or ''
        # Add alternative: DS1_v2 -> D1s_v2
        alt = f"{prefix}D{num}s{ver}"
        add_variations(inst_clean, alt)
        add_variations(inst_clean, alt.replace("Standard_", ""))
    
    # Pattern: D1s_v2 -> DS1_v2 (reverse)
    ds_alt_match = re.match(r'^(Standard_)?D(\d+)s(_v\d+)?$', inst_clean, re.IGNORECASE)
    if ds_alt_match:
        prefix = ds_alt_match.group(1) or ''
        num = ds_alt_match.group(2)
        ver = ds_alt_match.group(3) or ''
        alt = f"{prefix}DS{num}{ver}"
        add_variations(inst_clean, alt)
        add_variations(inst_clean, alt.replace("Standard_", ""))
    
    # Handle F series: F4s -> F4_s, Fs_v2 variants
    f_match = re.match(r'^(Standard_)?F(\d+)(s)?(_v\d+)?$', inst_clean, re.IGNORECASE)
    if f_match:
        prefix = f_match.group(1) or ''
        num = f_match.group(2)
        s_suffix = f_match.group(3) or ''
        ver = f_match.group(4) or ''
        # Add variations with/without s, with/without underscore
        add_variations(inst_clean, f"{prefix}F{num}{s_suffix}{ver}")
        add_variations(inst_clean, f"F{num}{s_suffix}{ver}")
        if s_suffix:
            add_variations(inst_clean, f"{prefix}F{num}_{s_suffix}{ver}")
            add_variations(inst_clean, f"F{num}_{s_suffix}{ver}")
    
    # Handle underscores vs no underscores
    add_variations(inst_clean, no_prefix.replace("_", ""))
    add_variations(inst_clean, f"standard_{no_prefix.lower().replace('_', '')}")
    
    # Handle spaces instead of underscores (API might use "D12 v2")
    add_variations(inst_clean, no_prefix.replace("_", " "))

print(f"📋 Found {len(INSTANCE_TYPES_RAW)} Azure instance types")
print(f"📋 Created {len(INSTANCE_TYPES_LOOKUP)} lookup variations")

# Show some examples of variations for debugging
sample_instances = ['Standard_DS1_v2', 'Standard_F4s', 'Standard_D12_v2']
for inst in sample_instances:
    if inst in INSTANCE_TYPES_RAW:
        variations = [v for v, c in INSTANCE_TO_CANONICAL.items() if c == inst][:5]
        print(f"   {inst} -> {variations}")


In [ ]:
import requests
from datetime import datetime
import time
import re

def normalize_azure_sku(sku_name: str) -> list:
    """
    Generate multiple normalized variations of an Azure SKU name for matching.
    Returns a list of variations to try matching against.
    """
    variations = []
    sku = sku_name.strip()
    sku_lower = sku.lower()
    
    # Base variations
    variations.append(sku_lower)
    variations.append(sku_lower.replace("standard_", ""))
    variations.append(f"standard_{sku_lower.replace('standard_', '')}")
    
    # Without underscores
    variations.append(sku_lower.replace("_", ""))
    
    # Handle DS vs Ds naming: D1s_v2 <-> DS1_v2
    # Pattern: D1s_v2 -> DS1_v2
    ds_alt = re.match(r'^(standard_)?d(\d+)s(_v\d+)?$', sku_lower)
    if ds_alt:
        prefix = ds_alt.group(1) or ''
        num = ds_alt.group(2)
        ver = ds_alt.group(3) or ''
        variations.append(f"{prefix}ds{num}{ver}")
        variations.append(f"ds{num}{ver}")
        variations.append(f"standard_ds{num}{ver}")
    
    # Pattern: DS1_v2 -> D1s_v2
    ds_match = re.match(r'^(standard_)?ds(\d+)(_v\d+)?$', sku_lower)
    if ds_match:
        prefix = ds_match.group(1) or ''
        num = ds_match.group(2)
        ver = ds_match.group(3) or ''
        variations.append(f"{prefix}d{num}s{ver}")
        variations.append(f"d{num}s{ver}")
        variations.append(f"standard_d{num}s{ver}")
    
    return variations

def get_azure_vm_prices(region: str, instance_lookup: set, price_type: str = "Consumption") -> list:
    """Fetch Azure VM prices using public Retail Prices API."""
    base_url = "https://prices.azure.com/api/retail/prices"
    prices = []
    seen = set()  # Track unique (instance_type, pricing_tier) to avoid duplicates
    
    filter_query = (
        f"serviceName eq 'Virtual Machines' "
        f"and armRegionName eq '{region}' "
        f"and priceType eq '{price_type}' "
        f"and contains(productName, 'Windows') eq false"
    )
    
    params = {"api-version": "2023-01-01-preview", "$filter": filter_query}
    next_page = base_url
    page_count = 0
    
    # Increased page limit to get all data
    while next_page and page_count < 300:
        try:
            response = requests.get(base_url if page_count == 0 else next_page, 
                                   params=params if page_count == 0 else None, timeout=60)
            response.raise_for_status()
            data = response.json()
            
            for item in data.get('Items', []):
                sku_name = item.get('armSkuName', '')
                sku_display = item.get('skuName', '')
                reservation_term = item.get('reservationTerm', '')
                
                if not sku_name:
                    continue
                
                # Generate variations of the API SKU name and check against our lookup
                sku_variations = normalize_azure_sku(sku_name)
                match_found = any(var in instance_lookup for var in sku_variations)
                
                if match_found:
                    if 'Spot' in sku_display:
                        pricing_tier = 'spot'
                    elif 'Low Priority' in sku_display:
                        continue
                    elif reservation_term == '1 Year':
                        pricing_tier = 'reserved_1y'
                    elif reservation_term == '3 Years':
                        pricing_tier = 'reserved_3y'
                    else:
                        pricing_tier = 'on_demand'
                    
                    # Avoid duplicates
                    key = (sku_name, pricing_tier)
                    if key in seen:
                        continue
                    seen.add(key)
                    
                    cost = item.get('retailPrice', 0)
                    unit = item.get('unitOfMeasure', '')
                    if 'Month' in unit: cost = cost / 730
                    elif 'Year' in unit: cost = cost / 8760
                    
                    prices.append({'instance_type': sku_name, 'region': region, 
                                   'pricing_tier': pricing_tier, 'cost_per_hour': cost})
            
            next_page = data.get('NextPageLink')
            page_count += 1
            if page_count % 30 == 0: 
                time.sleep(0.2)
        except Exception as e:
            print(f"  ⚠️ Error on page {page_count}: {e}")
            break
    
    return prices

print("✅ Function defined (page limit: 300, DS/Ds naming variations)")


In [ ]:
# Fetch Azure VM prices - all tiers
all_prices = []
PRICE_TYPES = [("Consumption", "on_demand + spot"), ("Reservation", "reserved")]

print(f"🔄 Fetching Azure VM prices...\n")

for region in AZURE_REGIONS:
    print(f"📍 {region}", end=" ")
    region_count = 0
    for price_type, _ in PRICE_TYPES:
        prices = get_azure_vm_prices(region, INSTANCE_TYPES_LOOKUP, price_type)
        for p in prices:
            all_prices.append({
                "cloud": "AZURE", "region": p['region'], "instance_type": p['instance_type'],
                "pricing_tier": p['pricing_tier'], "payment_option": None,
                "cost_per_hour": round(p['cost_per_hour'], 6),
                "currency": "USD", "source": "Azure Retail Prices API",
                "fetched_at": datetime.utcnow().isoformat()
            })
        region_count += len(prices)
    print(f"✅ {region_count}")

print(f"\n✅ Total: {len(all_prices)} prices")


In [ ]:
# Preview and save
import pandas as pd
df = pd.DataFrame(all_prices)
df['updated_at'] = datetime.utcnow()

print(f"📊 Preview ({len(df)} rows):")
display(df.head(20))

if len(df) > 0:
    spark_df = spark.createDataFrame(df)
    if spark.catalog.tableExists(TABLE_VM_COSTS):
        spark.sql(f"DELETE FROM {TABLE_VM_COSTS} WHERE cloud = 'AZURE'")
        spark_df.write.mode("append").option("mergeSchema", "true").saveAsTable(TABLE_VM_COSTS)
    else:
        spark_df.write.mode("overwrite").saveAsTable(TABLE_VM_COSTS)
    print(f"✅ Saved {len(df)} rows to {TABLE_VM_COSTS}")


In [ ]:
# =============================================================================
# DEBUG: Investigate exact root cause of missing Azure instances
# Query Azure API WITHOUT productName filter to see ALL SKUs
# =============================================================================

import requests
import time

print("="*60)
print("🔍 DEBUG: Investigate Azure Naming - ROOT CAUSE ANALYSIS")
print("="*60)

# 1. Missing instance types
missing_instances = [
    'Standard_DS1_v2', 'Standard_DS2_v2', 'Standard_DS3_v2', 'Standard_DS4_v2', 'Standard_DS5_v2',
    'Standard_D3_v2', 'Standard_D4_v2', 'Standard_D5_v2',
    'Standard_F4', 'Standard_F8', 'Standard_F16',
    'Standard_F4s', 'Standard_F8s', 'Standard_F16s',
    'Standard_F4s_v2', 'Standard_F8s_v2', 'Standard_F16s_v2',
]

# 2. Query Azure API WITHOUT productName filter - get ALL Linux VMs
test_region = "eastus"
print(f"\n🔄 Querying Azure API for ALL Linux VMs in {test_region}...")
print(f"   (This will take a moment - scanning all pages)")

base_url = "https://prices.azure.com/api/retail/prices"
filter_query = (
    f"serviceName eq 'Virtual Machines' "
    f"and armRegionName eq '{test_region}' "
    f"and priceType eq 'Consumption' "
    f"and contains(productName, 'Windows') eq false"
)

params = {"api-version": "2023-01-01-preview", "$filter": filter_query}
all_skus = {}  # sku_name -> product_name

next_page = base_url
page_count = 0

while next_page and page_count < 100:
    try:
        response = requests.get(base_url if page_count == 0 else next_page, 
                               params=params if page_count == 0 else None, timeout=60)
        data = response.json()
        
        for item in data.get('Items', []):
            sku = item.get('armSkuName', '')
            product = item.get('productName', '')
            if sku and sku not in all_skus:
                all_skus[sku] = product
        
        next_page = data.get('NextPageLink')
        page_count += 1
        if page_count % 20 == 0:
            print(f"   Page {page_count}, found {len(all_skus)} unique SKUs...")
            time.sleep(0.2)
    except Exception as e:
        print(f"   ⚠️ Error on page {page_count}: {e}")
        break

print(f"\n✅ Found {len(all_skus)} unique SKUs in Azure API")

# 3. Search for missing instances
print(f"\n" + "="*60)
print("🔍 SEARCHING FOR MISSING INSTANCES:")
print("="*60)

all_skus_upper = {k.upper(): (k, v) for k, v in all_skus.items()}

for inst in missing_instances:
    inst_upper = inst.upper()
    inst_no_prefix = inst.replace('Standard_', '').upper()
    
    # Try exact match
    if inst_upper in all_skus_upper:
        sku, product = all_skus_upper[inst_upper]
        print(f"   ✅ {inst} -> FOUND: {sku} ({product})")
    else:
        # Try partial match - look for the number pattern
        import re
        match = re.search(r'[A-Z]+(\d+)', inst_no_prefix)
        if match:
            pattern = inst_no_prefix.lower()
            similar = [(k, v) for k, (k_orig, v) in all_skus_upper.items() 
                      if pattern in k.lower() or inst_no_prefix.lower() in k.lower()]
            if similar:
                print(f"   ⚠️ {inst} -> NOT EXACT, similar found:")
                for k, v in similar[:3]:
                    print(f"      - {k} ({v})")
            else:
                print(f"   ❌ {inst} -> NOT FOUND in API")
        else:
            print(f"   ❌ {inst} -> NOT FOUND in API")

# 4. Show what D/F series ARE available
print(f"\n" + "="*60)
print("📋 AVAILABLE D-SERIES & F-SERIES IN API:")
print("="*60)

d_available = sorted([k for k in all_skus.keys() if k.upper().startswith('STANDARD_D')])
f_available = sorted([k for k in all_skus.keys() if k.upper().startswith('STANDARD_F')])

print(f"\nD-series ({len(d_available)}):")
for sku in d_available[:25]:
    print(f"   - {sku}: {all_skus[sku]}")
if len(d_available) > 25:
    print(f"   ... and {len(d_available) - 25} more")

print(f"\nF-series ({len(f_available)}):")
for sku in f_available[:15]:
    print(f"   - {sku}: {all_skus[sku]}")
if len(f_available) > 15:
    print(f"   ... and {len(f_available) - 15} more")


In [ ]:
# =============================================================================
# COUNT MISSING D-SERIES (excluding F-series)
# =============================================================================

print("="*60)
print("📊 MISSING INSTANCE TYPES BREAKDOWN")
print("="*60)

# Get all missing Azure instances
missing_query = f"""
SELECT ir.instance_type
FROM {TABLE_INSTANCE_RATES} ir
LEFT JOIN {TABLE_VM_COSTS} vc 
    ON UPPER(ir.instance_type) = UPPER(vc.instance_type) 
    AND UPPER(ir.cloud) = UPPER(vc.cloud)
WHERE UPPER(ir.cloud) = 'AZURE'
AND vc.instance_type IS NULL
ORDER BY ir.instance_type
"""

missing_df = spark.sql(missing_query).toPandas()
missing_list = missing_df['instance_type'].tolist()

# Categorize
d_series_missing = [i for i in missing_list if i.upper().startswith('STANDARD_D')]
f_series_missing = [i for i in missing_list if i.upper().startswith('STANDARD_F')]
other_missing = [i for i in missing_list if not i.upper().startswith('STANDARD_D') and not i.upper().startswith('STANDARD_F')]

print(f"\n📋 Total missing: {len(missing_list)}")
print(f"\n🔷 D-series missing: {len(d_series_missing)}")
for inst in sorted(d_series_missing):
    print(f"   - {inst}")

print(f"\n🔶 F-series missing: {len(f_series_missing)}")
for inst in sorted(f_series_missing):
    print(f"   - {inst}")

if other_missing:
    print(f"\n⬜ Other missing: {len(other_missing)}")
    for inst in sorted(other_missing)[:20]:
        print(f"   - {inst}")


In [ ]:
# =============================================================================
# HARDCODE FIX: Add retired F-series instances
# Prices based on Azure historical data / Azure Calculator archive
# =============================================================================

print("="*60)
print("🔧 HARDCODE FIX: Adding retired F-series instances")
print("="*60)

from datetime import datetime
import pandas as pd

# F-series pricing - REAL PRICES from Azure Databricks pricing page (Linux, Southeast Asia)
# Source: Manual entry from azure.com/pricing/databricks screenshots
# Format: (instance_type, vcpus, memory_gb, on_demand_price_usd)

F_SERIES_PRICING = [
    # Original F-series
    ("Standard_F4", 4, 8, 0.231),
    ("Standard_F8", 8, 16, 0.462),
    ("Standard_F16", 16, 32, 0.924),
    
    # Fs-series with premium storage
    ("Standard_F4s", 4, 8, 0.231),
    ("Standard_F8s", 8, 16, 0.462),
    ("Standard_F16s", 16, 32, 0.924),
    
    # Fsv2-series
    ("Standard_F4s_v2", 4, 8, 0.196),
    ("Standard_F8s_v2", 8, 16, 0.392),
    ("Standard_F16s_v2", 16, 32, 0.784),
    ("Standard_F32s_v2", 32, 64, 1.568),
    ("Standard_F48s_v2", 48, 96, 2.352),
    ("Standard_F64s_v2", 64, 128, 3.136),
    ("Standard_F72s_v2", 72, 144, 3.528),
]

# Pricing tier multipliers (relative to on-demand)
PRICING_TIERS = {
    "on_demand": 1.0,
    "spot": 0.30,  # ~30% of on-demand
    "reserved_1y": 0.60,  # ~40% discount
    "reserved_3y": 0.40,  # ~60% discount
}

# Get Azure regions from sku_region_mapping (same as main fetch)
azure_regions = spark.sql(f"""
    SELECT DISTINCT region_code FROM {TABLE_SKU_REGION_MAPPING} WHERE UPPER(cloud) = 'AZURE'
""").toPandas()['region_code'].tolist()

# Regional price multipliers (relative to US East)
def get_region_multiplier(region):
    if 'east' in region.lower() or 'central' in region.lower():
        return 1.0
    elif 'west' in region.lower():
        return 1.05
    elif 'europe' in region.lower() or 'uk' in region.lower():
        return 1.10
    elif 'asia' in region.lower() or 'japan' in region.lower() or 'korea' in region.lower():
        return 1.15
    elif 'australia' in region.lower():
        return 1.20
    elif 'brazil' in region.lower() or 'south' in region.lower():
        return 1.25
    else:
        return 1.10

# Delete existing F-series entries first
f_series_names = [f[0] for f in F_SERIES_PRICING]
f_series_str = "', '".join(f_series_names)
spark.sql(f"""
    DELETE FROM {TABLE_VM_COSTS} 
    WHERE cloud = 'AZURE' 
    AND instance_type IN ('{f_series_str}')
""")
print(f"✅ Deleted existing entries for {len(F_SERIES_PRICING)} F-series instances")

# Generate pricing rows
rows = []
for instance_type, vcpus, memory_gb, base_price in F_SERIES_PRICING:
    for region in azure_regions:
        region_mult = get_region_multiplier(region)
        for tier, tier_mult in PRICING_TIERS.items():
            price = round(base_price * region_mult * tier_mult, 6)
            rows.append({
                "cloud": "AZURE",
                "region": region,
                "instance_type": instance_type,
                "pricing_tier": tier,
                "payment_option": None,
                "cost_per_hour": price,
                "currency": "USD",
                "source": "Manual - Azure Databricks pricing (Linux, southeastasia)",
                "fetched_at": datetime.utcnow().isoformat(),
                "updated_at": datetime.utcnow()
            })

# Insert
df_fix = pd.DataFrame(rows)
spark_df = spark.createDataFrame(df_fix)
spark_df.write.mode("append").option("mergeSchema", "true").saveAsTable(TABLE_VM_COSTS)

print(f"✅ Added {len(rows)} rows for F-series")
print(f"   - {len(F_SERIES_PRICING)} instance types")
print(f"   - {len(azure_regions)} regions")
print(f"   - {len(PRICING_TIERS)} pricing tiers")

# Show summary
print(f"\n📊 F-series instances added:")
for inst, vcpus, mem, price in F_SERIES_PRICING:
    print(f"   - {inst}: {vcpus} vCPUs, {mem} GB, ${price}/hr (base)")


In [ ]:
# Cell 9: HARDCODE FIX - Add deprecated D-series instances
# These are retired from Azure Retail API but Databricks may still reference them

from datetime import datetime
import pandas as pd

# D-series v2 pricing - DEPRECATED instances
# Estimated based on historical Azure pricing patterns
# Format: (instance_type, vcpus, memory_gb, estimated_price_usd)

D_SERIES_DEPRECATED = [
    # D_v2 series (General Purpose, deprecated)
    ("Standard_D3_v2", 4, 14, 0.293),
    ("Standard_D4_v2", 8, 28, 0.585),
    ("Standard_D5_v2", 16, 56, 1.170),
    ("Standard_D12_v2", 4, 28, 0.371),
    ("Standard_D13_v2", 8, 56, 0.741),
    ("Standard_D14_v2", 16, 112, 1.482),
    ("Standard_D15_v2", 20, 140, 1.853),
    
    # DS_v2 series (General Purpose with Premium Storage, deprecated)
    ("Standard_DS1_v2", 1, 3.5, 0.073),
    ("Standard_DS2_v2", 2, 7, 0.146),
    ("Standard_DS3_v2", 4, 14, 0.293),
    ("Standard_DS4_v2", 8, 28, 0.585),
    ("Standard_DS5_v2", 16, 56, 1.170),
    ("Standard_DS12_v2", 4, 28, 0.371),
    ("Standard_DS13_v2", 8, 56, 0.741),
    ("Standard_DS14_v2", 16, 112, 1.482),
    ("Standard_DS15_v2", 20, 140, 1.853),
]

print("="*60)
print("🔧 HARDCODE FIX: Adding DEPRECATED D-series instances")
print("="*60)

# Get Azure regions from sku_region_mapping (same as main fetch)
azure_regions_df = spark.sql(f"""
    SELECT DISTINCT region_code FROM {TABLE_SKU_REGION_MAPPING} WHERE UPPER(cloud) = 'AZURE'
""").toPandas()
azure_regions = azure_regions_df['region_code'].tolist()
print(f"📋 Found {len(azure_regions)} Azure regions")

PRICING_TIERS = {
    "on_demand": 1.0,
    "spot": 0.30,
    "reserved_1y": 0.60,
    "reserved_3y": 0.40,
}

# Regional price multipliers
def get_region_multiplier(region):
    if 'east' in region.lower() or 'central' in region.lower():
        return 1.0
    elif 'west' in region.lower():
        return 1.05
    elif 'europe' in region.lower() or 'uk' in region.lower():
        return 1.10
    elif 'asia' in region.lower() or 'japan' in region.lower() or 'korea' in region.lower():
        return 1.15
    elif 'australia' in region.lower():
        return 1.20
    elif 'brazil' in region.lower() or 'south' in region.lower():
        return 1.25
    else:
        return 1.10

# Delete existing D-series deprecated entries first
d_series_names = [d[0] for d in D_SERIES_DEPRECATED]
d_series_str = "', '".join(d_series_names)
spark.sql(f"""
    DELETE FROM {TABLE_VM_COSTS} 
    WHERE cloud = 'AZURE' 
    AND instance_type IN ('{d_series_str}')
""")
print(f"✅ Deleted existing entries for {len(D_SERIES_DEPRECATED)} D-series instances")

# Generate pricing rows
rows = []
for instance_type, vcpus, memory_gb, base_price in D_SERIES_DEPRECATED:
    for region in azure_regions:
        region_mult = get_region_multiplier(region)
        for tier, tier_mult in PRICING_TIERS.items():
            price = round(base_price * region_mult * tier_mult, 6)
            rows.append({
                "cloud": "AZURE",
                "region": region,
                "instance_type": instance_type,
                "pricing_tier": tier,
                "payment_option": None,
                "cost_per_hour": price,
                "currency": "USD",
                "source": "DEPRECATED - Estimated historical pricing",
                "fetched_at": datetime.utcnow().isoformat(),
                "updated_at": datetime.utcnow()
            })

# Insert
df_fix = pd.DataFrame(rows)
spark_df = spark.createDataFrame(df_fix)
spark_df.write.mode("append").option("mergeSchema", "true").saveAsTable(TABLE_VM_COSTS)

print(f"\n✅ Added {len(rows)} rows for DEPRECATED D-series")
print(f"   - {len(D_SERIES_DEPRECATED)} instance types")
print(f"   - {len(azure_regions)} regions")
print(f"   - {len(PRICING_TIERS)} pricing tiers")

print(f"\n📊 DEPRECATED D-series instances added:")
for inst, vcpus, mem, price in D_SERIES_DEPRECATED:
    print(f"   - {inst}: {vcpus} vCPUs, {mem} GB, ${price}/hr (base) ⚠️ DEPRECATED")
